In [1]:
import ot
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import MiniBatchKMeans
import math
import scipy
from scipy.spatial import cKDTree
from scipy import integrate
from scipy import interpolate
from scipy.spatial import Delaunay
from scipy.spatial import Voronoi
from itertools import product
import torch
torch.manual_seed(100000)
import torch.nn as nn
import torch.optim as optim

np.random.seed(100000)  # NEW: keep the noisy data and sampled indices fixed

In [2]:
# settings
start = np.array((0., 1., 1.05))
dt = 0.01
subset_size = 20  # cell number
#slope = 0.5  # original weighting parameter (epsilon = 1/slope)
traj_length = int(1e7)
noise_level = 0.5
sample_size = int(500)
epsilon = 1e-4  # original teleportation parameter (not the weighting epsilon)
sampling_time = 1
simulation_t = int(2e4)
steps = 5
N_iters = 10000

# NEW: ablation settings
eps_values = np.linspace(0.5, 25, 10)#[1.0, 1.25, 1.5, 2.0, 2.5]#weighting parameter #2 in manuscript
mesh_shape = (2, 2, 5)  # 20 uniform centers
n_trials = 10 

In [4]:
# Lorenz-63 system and Euler simulation
def Lorenz(x):
    x = x.T
    dx = 10 * (x[1] - x[0])
    dy = x[0] * (28 - x[2]) - x[1]
    dz = x[0] * x[1] - (8/3) * x[2]
    f = np.array((dx, dy, dz)).T
    return f


def traj(length, starts=start, model=Lorenz):
    x = starts
    traj = np.zeros((length, len(starts)))
    traj[0] = np.array(x)
    for timestep in range(length-1):
        x = x + dt * model(x)
        traj[timestep+1] = np.array(x)
    return traj


def invariant_measure(matrix):
    N = len(matrix)
    rhs = (-1) * (epsilon/N) * torch.ones(N)
    rho = torch.linalg.solve(((1 - epsilon) * matrix - torch.eye(N)), rhs)
    return rho


# weighting functions
relu = nn.ReLU()
def decay(x):
    return relu(1 - slope*x)

def w(xs):
    dists = torch.cdist(xs, Voronoi_centers, p=2)
    pre_w = decay(dists)
    return pre_w / torch.sum(pre_w, dim=1, keepdim=True)


def Ulam(points, Tpoints):
    mat = torch.zeros((subset_size, subset_size))
    with torch.no_grad():
        randpts_idxs = torch.tensor(tree.query(points.detach().numpy())[1], dtype=torch.int)
    weights = w(Tpoints)
    mat.index_add_(0, randpts_idxs, weights)

    # CHANGE: avoid NaNs for empty uniform cells.
    row_mass = mat.sum(dim=1)
    mat = mat / row_mass.clamp_min(1e-12).unsqueeze(1)
    #print(mat.sum())
    #print(sum(mat))#should be the same for fair comparison#checked!
    return mat.T


#OT loss and backward
grid = np.arange(subset_size, dtype=float)
X, Y = np.meshgrid(grid, grid)
coords = np.stack([X.ravel(), Y.ravel()], axis=1)
FULL_OT_COST = ot.dist(coords, coords, metric="sqeuclidean")
class W2Loss(torch.autograd.Function):
    @staticmethod
    def forward(ctx, U_net):
        # Save input for backward
        U_net_np = U_net.detach().numpy()

        # Normalize inputs to sum to 1
        a = U_true_np.flatten()
        b = U_net_np.flatten()
        a = a / np.sum(a)
        b = b / np.sum(b)

        _, log = ot.emd(a, b, FULL_OT_COST, log=True)

        cost, grad = log['cost'], log['v']
        loss = cost
        grad_tensor = torch.tensor(grad, dtype=U_net.dtype).reshape_as(U_net)
        ctx.save_for_backward(grad_tensor)
        return torch.tensor(loss, dtype=U_net.dtype)

    @staticmethod
    def backward(ctx, grad_output):
        grad_tensor, = ctx.saved_tensors  # Unpack gradient
        return grad_output * grad_tensor.reshape(subset_size, subset_size) / subset_size

In [5]:
#noisy trajectory and normalized training data
trajectory_clean = traj(traj_length)
trajectory = trajectory_clean + np.random.normal(0, noise_level, ((traj_length, 3)))
trajectory = trajectory[int(1e4):]
del trajectory_clean#delete a variable binding to save memory

rand_idxs = np.random.choice(len(trajectory), size=sample_size, replace=False)
observed = trajectory[rand_idxs]
randpts = torch.tensor(trajectory[rand_idxs], dtype=torch.float)
Trandpts = torch.tensor(trajectory[rand_idxs+steps], dtype=torch.float)#5 steps

M_scale = torch.max(torch.abs(randpts))
randpts /= M_scale
Trandpts /= M_scale

#k-means construction, now stored once (new fixed random_state)
adaptive_centers = MiniBatchKMeans(
    n_clusters=subset_size
).fit(randpts).cluster_centers_

# NEW: uniform Cartesian centers, same number of cells
assert np.prod(mesh_shape) == subset_size#check if the numbers of cells are the same
low, high = randpts.numpy().min(axis=0), randpts.numpy().max(axis=0)#coord bounds
##plot(sanity check)
axes = [low[i] + (np.arange(n) + 0.5) * (high[i] - low[i]) / n
        for i, n in enumerate(mesh_shape)]                    #generate uniform cell centers
uniform_centers = np.array(list(product(*axes)), dtype=np.float32)#generate uniform mesh
meshes = {'adaptive': adaptive_centers, 'uniform': uniform_centers}

GT = traj(int(simulation_t), M_scale.item() * np.array(randpts[0]))
GTpts = torch.tensor((GT / M_scale.item())[:-steps], dtype=torch.float)
GTTpts = GT[steps:]
a = np.ones((simulation_t,)) / simulation_t

/homes/yinonghyn/.local/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:1930: FutureWarning: The default value of `n_init` will change from 3 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=3)


In [6]:
#axes

In [7]:
# define construction errors (w2, rmse)
# subsampling?
def reconstruction_error(net):
    net.eval()
    with torch.no_grad():
        x = torch.tensor(randpts[0], dtype=torch.float).reshape(1, 3)
        vals = [x.detach().numpy().flatten()]
        for _ in range(simulation_t-1):
            V_field = net(x)
            x = x + dt * V_field
            vals.append(x.detach().numpy().flatten())
        vals = np.array(vals)

        subGT, subvals = GT, np.array(M_scale.item() * vals)
        M = ot.dist(subGT, subvals)
        _, G0 = ot.emd(a, a, M, log=True)
        costW = (G0['cost'])**(1/2)

        netpts = GTpts
        for _ in range(steps):
            Vf = net(netpts)
            netpts = netpts + dt * Vf
        costL = np.mean((M_scale.item()*netpts.numpy() - GTTpts)**2)**(1/2)
    return costW, costL

In [8]:
# training with the new mesh + epsilon loops around net1.
results = {
    'adaptive': {eps: [] for eps in eps_values},
    'uniform': {eps: [] for eps in eps_values},
    'pointwise': []
}

for iteration in range(n_trials):
    seed = 100000 + iteration
    #different trials have different initializations, 
    #while all methods and weighting paramter within each trial share the same initialization.

    # Pointwise (baseline)
    torch.manual_seed(seed)
    net2 = nn.Sequential(
        nn.Linear(3, 100), nn.Tanh(),
        nn.Linear(100, 100), nn.Tanh(),
        nn.Linear(100, 100), nn.Tanh(),
        nn.Linear(100, 3)
    )
    optimizer2 = optim.Adam(net2.parameters(), lr=1e-3)
    net2.train()
    loss2 = []

    net2_randpts = randpts.clone()
    for _ in range(steps):
        V_field2 = net2(net2_randpts)
        net2_randpts = net2_randpts + dt * V_field2
    initial_L2 = torch.mean((net2_randpts - Trandpts)**2)

    for i in range(N_iters):
        optimizer2.zero_grad()
        net2_randpts = randpts
        for _ in range(steps):
            V_field2 = net2(net2_randpts)
            net2_randpts = net2_randpts + dt * V_field2
        L2 = torch.mean((net2_randpts - Trandpts)**2)
        # L2.backward()
        # optimizer2.step()
        # loss2.append(L2.item())
        if i % 500 == 0:
            if L2.item() < 0.02 * initial_L2:
                break
        L2.backward()
        optimizer2.step()
        loss2.append(L2.item())

    error = reconstruction_error(net2)
    results['pointwise'].append(error)
    print('trial', iteration, 'pointwise W2, RMSE:', error, flush=True)

    # NEW: compare the two fixed partitions at each weighting parameter
    for method, centers in meshes.items():
        Voronoi_centers = torch.tensor(centers, dtype=torch.float)
        tree = cKDTree(centers)
        for eps in eps_values:
            slope = 1 / eps
            with torch.no_grad():
                weights = w(Trandpts)

            U_true = Ulam(randpts, Trandpts)
            U_true_np = U_true.detach().cpu().numpy()


            # our method
            torch.manual_seed(seed)  # same initial parameters across eps and meshes
            net1 = nn.Sequential(
                nn.Linear(3, 100), nn.Tanh(),
                nn.Linear(100, 100), nn.Tanh(),
                nn.Linear(100, 100), nn.Tanh(),
                nn.Linear(100, 3)
            )
            optimizer1 = optim.Adam(net1.parameters(), lr=1e-3)
            net1.train()
            loss1 = []

            net1_randpts = randpts.clone()
            for _ in range(steps):
                V_field = net1(net1_randpts)
                net1_randpts = net1_randpts + dt * V_field
            U_net = Ulam(randpts, net1_randpts)
            initial_L1 = W2Loss.apply(U_net)

            for i in range(N_iters):
                optimizer1.zero_grad()
                net1_randpts = randpts
                for _ in range(steps):
                    V_field = net1(net1_randpts)
                    net1_randpts = net1_randpts + dt * V_field
                U_net = Ulam(randpts, net1_randpts)
                L1 = W2Loss.apply(U_net)
                # L1.backward()
                # optimizer1.step()
                # loss1.append(L1.item())
                with torch.no_grad():
                    if i % 500 == 0:
                        if L1.item() < 0.02 * initial_L1:
                            break
                L1.backward()
                optimizer1.step()
                loss1.append(L1.item())

            error = reconstruction_error(net1)
            results[method][eps].append(error)
            print('trial', iteration, method, 'eps =', eps,
                  'W2, RMSE:', error, flush=True)

/tmp/ipykernel_909344/3374839569.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(randpts[0], dtype=torch.float).reshape(1, 3)
/homes/yinonghyn/.local/lib/python3.10/site-packages/ot/lp/__init__.py:388: UserWarning: numItermax reached before optimality. Try to increase numItermax.
  result_code_string = check_result(result_code)


trial 0 pointwise W2, RMSE: (18.619787189074163, 0.6761350619540288)
tensor(20.)
tensor(20., grad_fn=<SumBackward0>)
tensor(20., grad_fn=<SumBackward0>)
tensor(20., grad_fn=<SumBackward0>)
tensor(20., grad_fn=<SumBackward0>)
tensor(20., grad_fn=<SumBackward0>)
tensor(20., grad_fn=<SumBackward0>)
tensor(20., grad_fn=<SumBackward0>)
tensor(20., grad_fn=<SumBackward0>)
tensor(20., grad_fn=<SumBackward0>)
tensor(20., grad_fn=<SumBackward0>)
tensor(20., grad_fn=<SumBackward0>)
tensor(20., grad_fn=<SumBackward0>)
tensor(20.0000, grad_fn=<SumBackward0>)
tensor(20., grad_fn=<SumBackward0>)
tensor(20., grad_fn=<SumBackward0>)
tensor(20., grad_fn=<SumBackward0>)
tensor(20., grad_fn=<SumBackward0>)
tensor(20., grad_fn=<SumBackward0>)
tensor(20.0000, grad_fn=<SumBackward0>)
tensor(20.0000, grad_fn=<SumBackward0>)
tensor(20., grad_fn=<SumBackward0>)
tensor(20.0000, grad_fn=<SumBackward0>)
tensor(20., grad_fn=<SumBackward0>)
tensor(20., grad_fn=<SumBackward0>)
tensor(20.0000, grad_fn=<SumBackward0>)

KeyboardInterrupt: 

In [ ]:
# NEW: Figure comparing three methods (0 = long-time W2, 1 = one-step RMSE)
metric = 0
plt.figure(figsize=(8, 5))
for method, label in [('adaptive', 'Unstructured mesh'), ('uniform', 'Uniform mesh')]:
    errors = [[r[metric] for r in results[method][eps]] for eps in eps_values]
    means = [np.mean(e) if e else np.nan for e in errors]
    stds = [np.std(e) if e else np.nan for e in errors]
    plt.errorbar(eps_values, means, yerr=stds, marker='o', capsize=3, label=label)

baseline = np.array(results['pointwise'])[:, metric]
plt.axhline(baseline.mean(), color='gray', linestyle='--', label='Pointwise')
plt.xlabel(r'Regularization parameter $\varepsilon$')
plt.ylabel(r'Long-time $W_2$ reconstruction error' if metric == 0 else 'One-step RMSE')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('lorenz63_epsilon_ablation.png', dpi=200)
plt.show()

In [ ]:
results

In [ ]:
errors

In [ ]:
metric = 1
plt.figure(figsize=(8, 5))
for method, label in [('adaptive', 'Unstructured mesh'), ('uniform', 'Uniform mesh')]:
    errors = [[r[metric] for r in results[method][eps]] for eps in eps_values]
    means = [np.mean(e) if e else np.nan for e in errors]
    stds = [np.std(e) if e else np.nan for e in errors]
    plt.errorbar(eps_values, means, yerr=stds, marker='o', capsize=3, label=label)

baseline = np.array(results['pointwise'])[:, metric]
plt.axhline(baseline.mean(), color='gray', linestyle='--', label='Pointwise')
plt.xlabel(r'Regularization parameter $\varepsilon$')
plt.ylabel(r'RMSE reconstruction error' if metric == 0 else 'One-step RMSE')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
#plt.savefig('lorenz63_epsilon_ablation.png', dpi=200)
plt.show()